# 01: EDA
### Machine Learning Competition

**Name:** Sara AlNajjar  
**Project 3**

## Imports

In [9]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

In [10]:
df = pd.read_csv('data.csv')
test = pd.read_csv('test.csv')
df.head()

,Property_id,Offer,URL,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent
0,6046,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3+ Maid,4,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0
1,2240,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0
2,6248,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,studio,1,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0
3,7177,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0
4,7842,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5+ Maid,5,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0


## 2. Shape & Unique

In [11]:
print('Train shape:', df.shape)
print('Test shape :', test.shape)

Train shape: (10578, 16)
Test shape : (3527, 15)


In [17]:
df['Beds'].unique()

<ArrowStringArray>
[     '3+ Maid',            '5',       'studio',            '3',
      '5+ Maid',            '2',      '4+ Maid',            '1',
      '2+ Maid',            '4',            '6',      '1+ Maid',
           '7+',      '6+ Maid', 'studio+ Maid',            '0',
     '7++ Maid',            '7',      '7+ Maid']
Length: 19, dtype: str

In [18]:
df['Baths'].unique()

<ArrowStringArray>
['4', '6', '1', '3', '5', '2', '7+', '7', 'none']
Length: 9, dtype: str

In [ ]:
df['Beds'].unique()

## 3. Dtypes

In [12]:
df.dtypes

Property_id            int64
Offer                    str
URL                      str
Property_type            str
Include_w_e              str
Title                    str
Area                     str
Governorate              str
Beds                     str
Baths                    str
Size                     str
Availability_date        str
Agent_name               str
Agency                   str
Amenities            float64
rent                 float64
dtype: object

## 4. Drop

In [19]:
drop_cols = ['URL', 'Title', 'Agent_name', 'Agency','Offer']
df = df.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)
df.columns.tolist()

['Property_id',
 'Property_type',
 'Include_w_e',
 'Area',
 'Governorate',
 'Beds',
 'Baths',
 'Size',
 'Availability_date',
 'Amenities',
 'rent']

## 5. Nulls & Fillna

In [6]:
df.isnull().sum()

Property_id            0
Property_type          0
Include_w_e            0
Area                   3
Governorate            3
Beds                   0
Baths                  0
Size                   0
Availability_date    519
Agency                 3
Amenities            306
rent                   3
dtype: int64

In [7]:
for blank in (df, test):
    blank['Governorate'] = blank['Governorate'].fillna('Unknown')
    blank['Area'] = blank['Area'].fillna('Unknown')
    blank['Agency'] = blank['Agency'].fillna('Unknown')
    blank['Availability_date'] = blank['Availability_date'].fillna('Unknown')
amenities_median = df['Amenities'].median()
df['Amenities'] = df['Amenities'].fillna(amenities_median)
test['Amenities'] = test['Amenities'].fillna(amenities_median)

df = df.dropna(subset=['rent'])
df.isnull().sum()

Property_id          0
Property_type        0
Include_w_e          0
Area                 0
Governorate          0
Beds                 0
Baths                0
Size                 0
Availability_date    0
Agency               0
Amenities            0
rent                 0
dtype: int64

## 6. Outliers